In [6]:
# -----------------------------
# STEP 1: Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from nltk.stem import WordNetLemmatizer
import re

# -----------------------------
# STEP 2: Load Dataset
# -----------------------------
print("Current Directory:", os.getcwd())
df = pd.read_csv("Customer_support_data.csv")
print("Dataset loaded successfully.\n")

# -----------------------------
# STEP 3: Show Columns in Table
# -----------------------------
columns_df = pd.DataFrame(df.columns, columns=['Column Names'])
print("Columns in Dataset:")
print(tabulate(columns_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Handle Missing Text Data
# -----------------------------
text_columns = ['Customer Remarks', 'Issue_reported at']
for col in text_columns:
    df[col] = df[col].fillna('')

# -----------------------------
# STEP 5: Inspect Sample Text
# -----------------------------
print("Sample Text Data:")
print(tabulate(df[text_columns].head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Text Preprocessing (No NLTK punkt)
# -----------------------------
stop_words = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself',
    'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this',
    'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to',
    'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
    'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some',
    'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too',
    'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now'
])

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not text or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove non-alphabetic characters
    tokens = text.split()  # simple split instead of word_tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing
df['Processed_Remarks'] = df['Customer Remarks'].apply(preprocess_text)

print("Processed Text Sample:")
print(tabulate(df[['Customer Remarks', 'Processed_Remarks']].head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 7: TF-IDF Feature Extraction
# -----------------------------
tfidf = TfidfVectorizer(max_features=100)
X_text = tfidf.fit_transform(df['Processed_Remarks'])

print("TF-IDF Feature Names Sample (first 20):")
print(tfidf.get_feature_names_out()[:20])
print("Shape of TF-IDF Matrix:", X_text.shape, "\n")

tfidf_df = pd.DataFrame(X_text.toarray(), columns=tfidf.get_feature_names_out(), index=df['Unique id'])
print("TF-IDF Features Sample (first 5 users):")
print(tabulate(tfidf_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 8: Combine CSAT / Product Category
# -----------------------------
numeric_features = df[['Unique id', 'Product_category', 'CSAT Score']].copy()

user_item_matrix = numeric_features.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score'
).fillna(0)

tfidf_df = tfidf_df.loc[user_item_matrix.index]

final_features = pd.concat([user_item_matrix.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

print("Final Features for Recommendation Model (Sample):")
print(tabulate(final_features.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 9: Summary
# -----------------------------
print("Week 11 NLP Integration Completed Successfully.")
print("Text Preprocessing, TF-IDF Feature Extraction, and integration with User-Item matrix done.")
print("Final dataset ready for model building in Week 12.")

# -----------------------------
# STEP 10: Save Week 11 Processed Features to CSV
# -----------------------------
output_path = 'week11_processed_features.csv'  # will save in Week 11 folder
final_features.to_csv(output_path, index=False)
print(f"Week 11 processed features saved successfully to: {output_path}")


Current Directory: f:\semester 7\DataScience_AI_Project\Week_11
Dataset loaded successfully.

Columns in Dataset:
╒════╤═════════════════════════╕
│    │ Column Names            │
╞════╪═════════════════════════╡
│  0 │ Unique id               │
├────┼─────────────────────────┤
│  1 │ channel_name            │
├────┼─────────────────────────┤
│  2 │ category                │
├────┼─────────────────────────┤
│  3 │ Sub-category            │
├────┼─────────────────────────┤
│  4 │ Customer Remarks        │
├────┼─────────────────────────┤
│  5 │ Order_id                │
├────┼─────────────────────────┤
│  6 │ order_date_time         │
├────┼─────────────────────────┤
│  7 │ Issue_reported at       │
├────┼─────────────────────────┤
│  8 │ issue_responded         │
├────┼─────────────────────────┤
│  9 │ Survey_response_Date    │
├────┼─────────────────────────┤
│ 10 │ Customer_City           │
├────┼─────────────────────────┤
│ 11 │ Product_category        │
├────┼──────────────────────